# La struttura della PA italiana

Analisi della struttura organizzativa della Pubblica Amministrazione italiana.

Dataset: `ipa_unita_organizzative`, `ipa_aree_organizzative_omogenee`, `ipa_enti`, `mef_partecipazioni`.

README: [`../README.md`](../README.md)

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

GCS = "gs://dataciviclab-clean"

for slug in ["ipa_unita_organizzative", "ipa_aree_organizzative_omogenee", "ipa_enti"]:
    path = f"{GCS}/{slug}/2026/{slug}_2026_clean.parquet"
    con.execute(f'CREATE VIEW "{slug}" AS SELECT * FROM "{path}"')

path_part = f"{GCS}/mef_partecipazioni/2023/mef_partecipazioni_2023_clean.parquet"
con.execute(f'CREATE VIEW mef_partecipazioni AS SELECT * FROM "{path_part}"')

C1, C2, C3, C4 = "#2c3e50", "#27ae60", "#e74c3c", "#95a5a6"

print("Dataset caricati:")
for slug in ["ipa_unita_organizzative", "ipa_aree_organizzative_omogenee", "ipa_enti", "mef_partecipazioni"]:
    cnt = con.execute(f"SELECT count(*) FROM \"{slug}\"").fetchone()[0]
    print(f"  {slug}: {cnt:,}")


## 1. Copertura generale

In [ ]:
df_cop = con.execute("""
SELECT 
  (SELECT COUNT(*) FROM ipa_unita_organizzative) AS tot_uo,
  (SELECT COUNT(*) FROM ipa_aree_organizzative_omogenee) AS tot_aoo,
  (SELECT COUNT(*) FROM ipa_enti) AS tot_enti,
  (SELECT COUNT(DISTINCT codice_ipa) FROM ipa_unita_organizzative) AS enti_con_uo,
  (SELECT COUNT(*) FROM ipa_unita_organizzative WHERE codice_uni_uo_padre IS NOT NULL AND codice_uni_uo_padre != '') AS uo_con_padre
""").fetchdf()
df_cop

## 2. Enti per macrocategoria

In [ ]:
df_cat = con.execute("""
SELECT 
  CASE 
    WHEN codice_categoria IN ('L6','L8','L12','L18','L36') THEN 'Enti locali'
    WHEN codice_categoria IN ('L33') THEN 'Scuole'
    WHEN codice_categoria IN ('L7','L34') THEN 'Sanita'
    WHEN codice_categoria IN ('L5') THEN 'Regioni'
    WHEN codice_categoria IN ('C1','C13','C9') THEN 'Stato centrale'
    WHEN codice_categoria IN ('C14') THEN 'Ordini professionali'
    WHEN codice_categoria IN ('L37','SA','SAG','S01') THEN 'Societa e gestori'
    ELSE 'Altri enti'
  END AS macrocategoria,
  COUNT(*) AS enti,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM ipa_enti), 1) AS pct
FROM ipa_enti
GROUP BY macrocategoria
ORDER BY enti DESC
""").fetchdf()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_cat['macrocategoria'], df_cat['enti'], color=C1, height=0.6)
for bar, pct in zip(bars, df_cat['pct']):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f"{int(bar.get_width()):,} ({pct}%)", va='center', fontsize=10, color=C4)
ax.set_xlabel('Enti')
ax.set_title('Enti PA per macrocategoria', fontsize=14, fontweight='bold', color=C1)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'enti_per_macrocategoria.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Top 15 enti per complessita organizzativa

In [ ]:
df_top = con.execute("""
SELECT 
  uo.codice_ipa,
  uo.denominazione_ente,
  COUNT(*) AS num_uo
FROM ipa_unita_organizzative uo
GROUP BY uo.codice_ipa, uo.denominazione_ente
ORDER BY num_uo DESC
LIMIT 15
""").fetchdf()

df_top['denominazione_ente'] = df_top['denominazione_ente'].str.replace('"', '')
df_top['label'] = df_top.apply(lambda r: r['denominazione_ente'][:50] + '...' if len(r['denominazione_ente']) > 50 else r['denominazione_ente'], axis=1)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(df_top)), df_top['num_uo'], color=C1, height=0.6)
for i, (_, row) in enumerate(df_top.iterrows()):
    ax.text(row['num_uo'] + 50, i, f"{row['num_uo']:,}", va='center', fontsize=9, color=C4)
ax.set_yticks(range(len(df_top)))
ax.set_yticklabels(df_top['label'], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Unita Organizzative')
ax.set_title('Top 15 enti per numero di unita organizzative', fontsize=14, fontweight='bold', color=C1)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'top_enti_complessita.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Distribuzione profondita gerarchica

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE livelli_uo AS
WITH RECURSIVE tree AS (
  SELECT codice_uni_uo, codice_ipa, 1 AS livello
  FROM ipa_unita_organizzative
  WHERE codice_uni_uo_padre IS NULL OR codice_uni_uo_padre = ''
  UNION ALL
  SELECT uo.codice_uni_uo, uo.codice_ipa, tree.livello + 1
  FROM ipa_unita_organizzative uo
  JOIN tree ON uo.codice_uni_uo_padre = tree.codice_uni_uo
  WHERE uo.codice_uni_uo_padre IS NOT NULL AND uo.codice_uni_uo_padre != ''
)
SELECT * FROM tree
""")

df_liv = con.execute("""
SELECT livello, COUNT(*) AS count
FROM livelli_uo
GROUP BY livello
ORDER BY livello
""").fetchdf()

colors = ['#2c3e50', '#27ae60', '#e74c3c', '#3498db', '#9b59b6', '#e67e22', '#1abc9c', '#e74c3c', '#95a5a6'][:len(df_liv)]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df_liv['livello'].astype(str), df_liv['count'], color=colors)
for bar in bars:
    height = bar.get_height()
    pct = height / df_liv['count'].sum() * 100
    ax.text(bar.get_x() + bar.get_width()/2, height + 500,
            f'{int(height):,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, color='#2c3e50')
ax.set_xlabel('Livello gerarchico')
ax.set_ylabel('Unita Organizzative')
ax.set_title('Distribuzione dei livelli gerarchici nella PA', fontsize=14, fontweight='bold', color='#2c3e50')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'profondita_gerarchica.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Uffici piu comuni

In [ ]:
df_uffici = con.execute("""
SELECT descrizione_uo, COUNT(*) AS count
FROM ipa_unita_organizzative
WHERE descrizione_uo NOT IN ('Uff_eFatturaPA', 'Ufficio per la transizione al Digitale')
GROUP BY descrizione_uo
ORDER BY count DESC
LIMIT 15
""").fetchdf()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(df_uffici)), df_uffici['count'], color=C1, height=0.6)
for i, (_, row) in enumerate(df_uffici.iterrows()):
    ax.text(row['count'] + 5, i, f"{row['count']}", va='center', fontsize=9, color=C4)
ax.set_yticks(range(len(df_uffici)))
ax.set_yticklabels(df_uffici['descrizione_uo'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Unita Organizzative')
ax.set_title('Uffici piu comuni nella PA (esclusi RTD e fatturazione)', fontsize=14, fontweight='bold', color=C1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'uffici_piu_comuni.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Vertici per macrocategoria

In [ ]:
df_vertici = con.execute("""
WITH vertici_raw AS (
  SELECT 
    CASE 
      WHEN codice_categoria IN ('L6','L8','L12','L18','L36') THEN 'Enti locali'
      WHEN codice_categoria IN ('L33') THEN 'Scuole'
      WHEN codice_categoria IN ('L7','L34') THEN 'Sanita'
      WHEN codice_categoria IN ('L5') THEN 'Regioni'
      WHEN codice_categoria IN ('C1','C13','C9') THEN 'Stato centrale'
      ELSE 'Altro'
    END AS macrocategoria,
    titolo_responsabile,
    COUNT(*) AS count
  FROM ipa_enti
  WHERE titolo_responsabile IS NOT NULL AND titolo_responsabile != ''
  GROUP BY macrocategoria, titolo_responsabile
)
SELECT macrocategoria, titolo_responsabile, count
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY macrocategoria ORDER BY count DESC) AS rn
  FROM vertici_raw
) WHERE rn <= 3
ORDER BY macrocategoria, count DESC
""").fetchdf()

categorie = df_vertici['macrocategoria'].unique()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for idx, cat in enumerate(categorie):
    ax = axes[idx]
    group = df_vertici[df_vertici['macrocategoria'] == cat]
    colors_bar = ['#2c3e50', '#27ae60', '#e74c3c'][:len(group)]
    ax.barh(group['titolo_responsabile'], group['count'], color=colors_bar, height=0.5)
    for _, row in group.iterrows():
        ax.text(row['count'] + 20, list(group['titolo_responsabile']).index(row['titolo_responsabile']),
                f'{int(row["count"]):,}', va='center', fontsize=8, color='#95a5a6')
    ax.set_title(cat, fontsize=11, fontweight='bold', color='#2c3e50')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for idx in range(len(categorie), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Vertici della PA per macrocategoria', fontsize=14, fontweight='bold', color='#2c3e50')
plt.tight_layout()
plt.savefig(FIG_DIR / 'vertici_per_tipologia.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Partecipazioni pubbliche

In [ ]:
df_part = con.execute("""
SELECT 
  amministrazione_macrocategoria,
  COUNT(DISTINCT partecipata_codice_fiscale) AS societa_partecipate
FROM mef_partecipazioni
GROUP BY amministrazione_macrocategoria
ORDER BY societa_partecipate DESC
""").fetchdf()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(df_part)), df_part['societa_partecipate'], color=C1, height=0.6)
for i, (_, row) in enumerate(df_part.iterrows()):
    ax.text(row['societa_partecipate'] + 50, i, f"{int(row['societa_partecipate']):,}", va='center', fontsize=9, color=C4)
ax.set_yticks(range(len(df_part)))
ax.set_yticklabels(df_part['amministrazione_macrocategoria'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Societa partecipate')
ax.set_title('Partecipazioni pubbliche per macrocategoria', fontsize=14, fontweight='bold', color=C1)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'mappa_partecipate.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Tabella riassuntiva per macrocategoria

In [ ]:
df_rias = con.execute("""
WITH enti_cat AS (
  SELECT codice_ipa, 
    CASE 
      WHEN codice_categoria IN ('L6','L8','L12','L18','L36') THEN 'Enti locali'
      WHEN codice_categoria IN ('L33') THEN 'Scuole'
      WHEN codice_categoria IN ('L7','L34') THEN 'Sanita'
      WHEN codice_categoria IN ('L5') THEN 'Regioni'
      WHEN codice_categoria IN ('C1','C13','C9') THEN 'Stato centrale'
      WHEN codice_categoria IN ('C14') THEN 'Ordini professionali'
      ELSE 'Altro'
    END AS macrocategoria
  FROM ipa_enti
)
SELECT e.macrocategoria,
  COUNT(DISTINCT uo.codice_uni_uo) AS tot_uo,
  COUNT(DISTINCT uo.codice_ipa) AS enti_con_uo,
  ROUND(AVG(liv.livello), 2) AS profondita_media
FROM enti_cat e
LEFT JOIN ipa_unita_organizzative uo ON e.codice_ipa = uo.codice_ipa
LEFT JOIN livelli_uo liv ON uo.codice_uni_uo = liv.codice_uni_uo
GROUP BY e.macrocategoria
ORDER BY tot_uo DESC
""").fetchdf()
df_rias

## 9. Presenza uffici: quanti enti hanno ciascun ufficio

In [ ]:
df_pres = con.execute("""
SELECT 
  descrizione_uo,
  COUNT(DISTINCT codice_ipa) AS enti_con_ufficio
FROM ipa_unita_organizzative
WHERE descrizione_uo NOT IN ('Uff_eFatturaPA', 'Ufficio per la transizione al Digitale')
  AND codice_ipa IN (SELECT codice_ipa FROM ipa_enti)
GROUP BY descrizione_uo
ORDER BY enti_con_ufficio DESC
LIMIT 15
""").fetchdf()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(df_pres)), df_pres['enti_con_ufficio'], color=C2, height=0.6)
for i, (_, row) in enumerate(df_pres.iterrows()):
    ax.text(row['enti_con_ufficio'] + 10, i, f"{int(row['enti_con_ufficio'])}", va='center', fontsize=9, color=C4)
ax.set_yticks(range(len(df_pres)))
ax.set_yticklabels(df_pres['descrizione_uo'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Enti che hanno questo ufficio')
ax.set_title('Quanti enti hanno ciascun ufficio?', fontsize=14, fontweight='bold', color=C1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'presenza_uffici.png', dpi=150, bbox_inches='tight')
plt.show()